In [194]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
import pickle # pickle is used to save the machine learning model file
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

# first we have to create a virtual environment so in terminal type the below commands
# python -m venv loanapp
# loanapp/Scripts/activate

# in requirements.txt file we have all the required modules
# To install all the modules of requirement.txt we have to type:
# pip install -r requirement.txt

In [195]:
df=pd.read_csv("Loan_data.csv")
df["TotalIncome"] = df["ApplicantIncome"] + df["CoapplicantIncome"]
df["Income_Loan_Ratio"] = df["TotalIncome"] / (df["LoanAmount"] + 1)
df["EMI"] = df["LoanAmount"] / (df["Loan_Amount_Term"] + 1)

df.head()
#df.shape

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,TotalIncome,Income_Loan_Ratio,EMI
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y,5849.0,NaN,NaN
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N,6091.0,47.217054,0.354571
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y,3000.0,44.776119,0.182825
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y,4941.0,40.834711,0.332410
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y,6000.0,42.253521,0.390582


In [196]:
y=df["Loan_Status"] # Target column in y
x=df.drop(columns=["Loan_Status","Loan_ID"]) # All the columns except Target column in X

In [197]:
categorical_cols=x.select_dtypes(include=['object']).columns.tolist()
numerical_cols=x.select_dtypes(include=['int64','float64']).columns.tolist()

print("The categorical columns are :",categorical_cols)
print("The numerical columns are :",numerical_cols)

The categorical columns are : ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
The numerical columns are : ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'TotalIncome', 'Income_Loan_Ratio', 'EMI']


In [198]:
# convert categorical columns to numerical there are vales like yes or no here we cannot rank them so we will use one-hot-encoding
# for numerical values we have to standardscalar to ensure that the values are in the same scale range
# we sue column transformer to preform preprocessing in a single command as given below
preprocessing = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols),

        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler())
        ]), numerical_cols)
    ]
)

In [199]:
# pipeline combines these preprocessing steps and the ml algorithm we are going to use here we are using randomforest as it is a classification problem
model=Pipeline([ 
    ("preprocessing",preprocessing),
    ("rf",RandomForestClassifier(n_estimators=150,
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42))
])
# As we used pipeline now both preprocessing and training of the model both will occur at the same time 
# usually we write like model=RandomForestClassifier() and then model.fit(x_train,y_train) but here we are writing it as a pipe line first then we fit it

In [200]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(model, x, y, cv=5)

print(f"Cross-validation accuracy: {cv_scores.mean():.4f}")
print(f"CV Std Dev: {cv_scores.std():.4f}")

Cross-validation accuracy: 0.8062
CV Std Dev: 0.0226


In [201]:
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

model.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [202]:
from sklearn.metrics import confusion_matrix,classification_report

# Prediction on test set
y_pred=model.predict(X_test)

print("Model Evaluation on test set")
print("-----------------------------")

#Accuracy

acc=accuracy_score(y_test,y_pred)
print(f"Accuracy: {acc:.2%}\n")

train_pred = model.predict(X_train)
print("Train Accuracy:", accuracy_score(y_train, train_pred))

#Confusion matrix
print("Confusion Matrix")
print(confusion_matrix(y_test,y_pred))
print("\n")

#Classification report
print("Classification Report :")
print(classification_report(y_test,y_pred))



Model Evaluation on test set
-----------------------------
Accuracy: 81.30%

Train Accuracy: 0.8757637474541752
Confusion Matrix
[[21 22]
 [ 1 79]]


Classification Report :
              precision    recall  f1-score   support

           N       0.95      0.49      0.65        43
           Y       0.78      0.99      0.87        80

    accuracy                           0.81       123
   macro avg       0.87      0.74      0.76       123
weighted avg       0.84      0.81      0.79       123



In [203]:
# Now we are trying this for a sample dataset created by us
sample = pd.DataFrame([{
    "Gender": "Male",
    "Married": "Yes",
    "Dependents": "1",
    "Education": "Graduate",
    "Self_Employed": "No",
    "ApplicantIncome": 4500,
    "CoapplicantIncome": 3000,
    "LoanAmount": 128,
    "Loan_Amount_Term": 360,
    "Credit_History": 1,
    "Property_Area": "Urban"
}])

# 🔥 APPLY SAME FEATURE ENGINEERING

sample["TotalIncome"] = sample["ApplicantIncome"] + sample["CoapplicantIncome"]
sample["Income_Loan_Ratio"] = sample["TotalIncome"] / (sample["LoanAmount"] + 1)
sample["EMI"] = sample["LoanAmount"] / (sample["Loan_Amount_Term"] + 1)

print("Prediction:", model.predict(sample)[0])
print("Probability:", model.predict_proba(sample)[0][1])

Prediction:

 Y
Probability: 0.7717926460395464


In [204]:
with open("loan_model.pkl","wb") as f: # we are saving everything in a file called loan_model.pkl and wb means write binary
    pickle.dump(model,f)